# Heterogeneous Propagation Medium Example

Ported from: k-Wave/examples/example_ivp_heterogeneous_medium.m

Simulates the pressure field generated by an initial pressure distribution
(two discs) within a two-dimensional heterogeneous propagation medium with
spatially varying sound speed and density. It builds on the Homogeneous
Propagation Medium Example.

The left half of the domain has a higher sound speed (1800 m/s vs 1500 m/s)
and the right three-quarters has a higher density (1200 kg/m^3 vs 1000 kg/m^3).
You should observe wavefront distortion and partial reflections at the
impedance boundaries.

In [1]:
import numpy as np

from kwave.data import Vector
from kwave.kgrid import kWaveGrid
from kwave.kmedium import kWaveMedium
from kwave.ksensor import kSensor
from kwave.ksource import kSource
from kwave.kspaceFirstOrder import kspaceFirstOrder
from kwave.utils.mapgen import make_cart_circle, make_disc

In [2]:
def setup():
    """Set up the simulation physics (grid, medium, source).

    Returns:
        tuple: (kgrid, medium, source)
    """

    # create the computational grid
    Nx = 128  # number of grid points in the x direction
    Ny = 128  # number of grid points in the y direction
    dx = 0.1e-3  # grid point spacing in the x direction [m]
    dy = 0.1e-3  # grid point spacing in the y direction [m]
    kgrid = kWaveGrid(Vector([Nx, Ny]), Vector([dx, dy]))

    # define the properties of the propagation medium
    c = 1500 * np.ones((Nx, Ny))  # [m/s]
    c[:64, :] = 1800  # MATLAB: 1:Nx/2
    rho = 1000 * np.ones((Nx, Ny))  # [kg/m^3]
    rho[:, 31:] = 1200  # MATLAB: Ny/4:Ny = 32:128 (1-based) -> 31: (0-based)
    medium = kWaveMedium(sound_speed=c, density=rho)

    # create initial pressure distribution using make_disc
    # -- disc 1 --
    disc_1_magnitude = 5  # [Pa]
    disc_1_x_pos = 50  # [grid points, 1-based]
    disc_1_y_pos = 50  # [grid points, 1-based]
    disc_1_radius = 8  # [grid points]
    disc_1 = disc_1_magnitude * make_disc(Vector([Nx, Ny]), Vector([disc_1_x_pos, disc_1_y_pos]), disc_1_radius)

    # -- disc 2 --
    disc_2_magnitude = 3  # [Pa]
    disc_2_x_pos = 80  # [grid points, 1-based]
    disc_2_y_pos = 60  # [grid points, 1-based]
    disc_2_radius = 5  # [grid points]
    disc_2 = disc_2_magnitude * make_disc(Vector([Nx, Ny]), Vector([disc_2_x_pos, disc_2_y_pos]), disc_2_radius)

    source = kSource()
    source.p0 = (disc_1 + disc_2).astype(float)

    # set time array (pass full c array — makeTime uses max internally for dt)
    kgrid.makeTime(c)

    return kgrid, medium, source

In [3]:
def run(backend="python", device="cpu", quiet=True):
    """Run with the original Cartesian circular sensor.

    Returns:
        dict: Simulation results with key 'p' (sensor_points x time_steps).
    """
    kgrid, medium, source = setup()

    # define a centered circular sensor (Cartesian) matching MATLAB original
    sensor_radius = 4e-3  # [m]
    num_sensor_points = 50
    sensor_mask = make_cart_circle(sensor_radius, num_sensor_points)
    sensor = kSensor(mask=sensor_mask)

    return kspaceFirstOrder(
        kgrid,
        medium,
        source,
        sensor,
        backend=backend,
        device=device,
        quiet=quiet,
        pml_inside=True,
    )

In [4]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    result = run(quiet=False)
    p = np.asarray(result["p"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # plot the simulated sensor data
    ax = axes[0]
    im = ax.imshow(p, aspect="auto", vmin=-1, vmax=1)
    ax.set_ylabel("Sensor Position")
    ax.set_xlabel("Time Step")
    ax.set_title("Sensor Data")
    fig.colorbar(im, ax=ax)

    # plot the initial pressure with medium overlay
    kgrid, medium, source = setup()
    Nx, Ny = 128, 128
    ax = axes[1]
    extent = [
        kgrid.y_vec[0] * 1e3,
        kgrid.y_vec[-1] * 1e3,
        kgrid.x_vec[-1] * 1e3,
        kgrid.x_vec[0] * 1e3,
    ]
    # show sound speed as background
    ax.imshow(medium.sound_speed, extent=extent, cmap="gray", alpha=0.4)
    # overlay initial pressure
    p0 = source.p0.copy()
    p0_masked = np.ma.masked_where(p0 == 0, p0)
    ax.imshow(p0_masked, extent=extent, cmap="hot", vmin=0, vmax=5)
    ax.set_xlabel("y-position [mm]")
    ax.set_ylabel("x-position [mm]")
    ax.set_title("Initial Pressure + Sound Speed")
    ax.set_aspect("equal")

    fig.suptitle("Heterogeneous Propagation Medium Example")
    fig.tight_layout()
    plt.show()

k-Wave:   0%|          | 0/725 [00:00<?, ?step/s]

k-Wave:  15%|█▍        | 108/725 [00:00<00:00, 1075.01step/s]

k-Wave:  30%|███       | 218/725 [00:00<00:00, 1085.59step/s]

k-Wave:  45%|████▌     | 327/725 [00:00<00:00, 1086.32step/s]

k-Wave:  61%|██████    | 441/725 [00:00<00:00, 1104.28step/s]

k-Wave:  77%|███████▋  | 555/725 [00:00<00:00, 1115.01step/s]

k-Wave:  92%|█████████▏| 669/725 [00:00<00:00, 1121.72step/s]

k-Wave: 100%|██████████| 725/725 [00:00<00:00, 1109.56step/s]

/var/folders/c_/04qn6kmn1h3d0nqkbjw264wc0000gp/T/ipykernel_73427/4130718478.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
